# Multinomial Logit: Discrete Choice with Heterogeneous Preferences

**Links:**
- [GitHub](https://github.com/rawatpranjal/deep-inference)
- [PyPI](https://pypi.org/project/deep-inference/)
- [Documentation](https://rawatpranjal.github.io/deep-inference/)

**References:**
- Farrell, Liang, Misra (2021) "Deep Neural Networks for Estimation and Inference" *Econometrica*
- Farrell, Liang, Misra (2025) "Deep Learning for Individual Heterogeneity"
- Hetzenecker, Osterhaus (2024) "Deep Learning for Heterogeneous Parameters in Discrete Choice Models" *arXiv:2408.09560*

---

This notebook validates `structural_dml` for **multinomial logit** (conditional logit / McFadden) models.

## Model

$$V_{ij} = \alpha_j(W) + X'_{ij} \cdot \beta(W), \quad P(Y=j \mid W, X) = \text{softmax}(V)[j]$$

- $W$: individual characteristics (NN input)
- $X_{ij}$: alternative-specific attributes
- $\alpha_0 = 0$ (reference alternative)
- $\theta = [\alpha_1, \ldots, \alpha_{J-1}, \beta_1, \ldots, \beta_K]$

## Data Encoding

| Variable | Shape | Description |
|----------|-------|-------------|
| `W` (as `X`) | `(n, d_w)` | Individual characteristics |
| `T` | `(n, J*K)` | Packed alternative attributes |
| `Y` | `(n,)` | Chosen alternative (float: 0, 1, ..., J-1) |

**Target:** $\mu^* = E[\beta_1(W)] = -0.8$

## Section 1: Setup & DGP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax
import warnings
import sys
from pathlib import Path

# Use local deep_inference
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from deep_inference import structural_dml

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

print("Setup complete!")

In [ ]:
# DGP Parameters
J = 3   # alternatives
K = 2   # attributes per alternative
d_w = 3 # individual characteristics

# Alpha intercepts: alpha_j(W) = a0_j + a1_j * W[0]
a0 = [0.0, 0.5, -0.3]   # alpha_0 = 0 (reference)
a1 = [0.0, 0.2, -0.1]

# Beta coefficients: beta_k(W) = b0_k + b1_k * W[0]
b0 = [-0.8, 0.5]
b1 = [-0.2, 0.1]

MU_TRUE = b0[0]  # E[beta_1(W)] = -0.8 since E[W[0]] = 0

print(f"DGP: P(Y=j) = softmax(V)[j]")
print(f"  V_ij = alpha_j(W) + X'_ij * beta(W)")
print(f"  J={J} alternatives, K={K} attributes, d_w={d_w}")
print(f"  alpha_1(W) = {a0[1]} + {a1[1]}*W[0]")
print(f"  alpha_2(W) = {a0[2]} + {a1[2]}*W[0]")
print(f"  beta_1(W) = {b0[0]} + {b1[0]}*W[0]")
print(f"  beta_2(W) = {b0[1]} + {b1[1]}*W[0]")
print(f"  theta = [alpha_1, alpha_2, beta_1, beta_2] (dim={J-1+K})")
print(f"  Target mu* = E[beta_1(W)] = {MU_TRUE}")

In [ ]:
def generate_data(n, seed=None):
    """Generate data from multinomial logit DGP."""
    if seed is not None:
        np.random.seed(seed)
    
    # Individual characteristics W ~ N(0, I)
    W = np.random.normal(0, 1, (n, d_w))
    
    # True parameters
    alphas = np.column_stack([a0[j] + a1[j] * W[:, 0] for j in range(J)])
    betas = np.column_stack([b0[k] + b1[k] * W[:, 0] for k in range(K)])
    theta_true = np.column_stack([alphas[:, 1:], betas])  # exclude alpha_0
    
    # Alternative attributes X ~ N(0, 1)
    X_alt = np.random.normal(0, 1, (n, J, K))
    
    # Utilities
    V = alphas.copy()
    for j in range(J):
        V[:, j] += np.sum(X_alt[:, j, :] * betas, axis=1)
    probs = softmax(V, axis=1)
    
    # Sample choices
    Y = np.array([np.random.choice(J, p=probs[i]) for i in range(n)]).astype(float)
    
    # Pack T: (n, J, K) -> (n, J*K)
    T = X_alt.reshape(n, -1)
    
    return {'Y': Y, 'T': T, 'W': W, 'theta_true': theta_true,
            'alphas': alphas, 'betas': betas, 'probs': probs}

# Generate data
n = 5000
data = generate_data(n, seed=42)

print(f"Shapes: Y={data['Y'].shape}, T={data['T'].shape}, W={data['W'].shape}")
print(f"theta_true shape: {data['theta_true'].shape} (= [alpha_1, alpha_2, beta_1, beta_2])")
print(f"True mu* = E[beta_1(W)] = {MU_TRUE}")
print(f"Sample mean beta_1: {data['betas'][:, 0].mean():.4f}")
print(f"Y distribution: {[f'alt {j}: {(data["Y"]==j).mean():.1%}' for j in range(J)]}")

## Section 2: Run Structural DML

In [ ]:
print(f"Running structural_dml with multinomial_logit...")
print(f"  n={n}, J={J}, K={K}, theta_dim={J-1+K}")
print(f"  patience=50, epochs=200, n_folds=20")
print()

result = structural_dml(
    Y=data['Y'],
    T=data['T'],
    X=data['W'],
    family='multinomial_logit',
    n_alternatives=J,
    n_attributes=K,
    hidden_dims=[64, 32],
    epochs=200,
    patience=50,
    n_folds=20,
    lr=0.01,
    verbose=False
)

print(result.summary())

## Section 3: Compare to Oracle

In [ ]:
# Compare estimate to true value
print("="*60)
print("COMPARISON: NN vs Oracle")
print("="*60)
print(f"{'Metric':<25} {'Value':<15}")
print("-"*40)
print(f"{'True mu*':<25} {MU_TRUE:<15.4f}")
print(f"{'NN mu_hat (debiased)':<25} {result.mu_hat:<15.4f}")
print(f"{'NN mu_naive':<25} {result.mu_naive:<15.4f}")
print(f"{'SE (IF)':<25} {result.se:<15.4f}")
print(f"{'95% CI':<25} [{result.ci_lower:.4f}, {result.ci_upper:.4f}]")
covers = result.ci_lower <= MU_TRUE <= result.ci_upper
print(f"{'Covers true mu*?':<25} {covers}")
print(f"{'Bias':<25} {result.mu_hat - MU_TRUE:<15.4f}")
print("="*60)

## Section 4: Parameter Recovery

In [ ]:
# Compute correlations between estimated and true parameters
theta_hat = result.theta_hat  # (n, 4): [alpha_1, alpha_2, beta_1, beta_2]
theta_true = data['theta_true']

param_names = ['alpha_1', 'alpha_2', 'beta_1', 'beta_2']

print("="*60)
print("PARAMETER RECOVERY")
print("="*60)
print(f"{'Component':<12} {'RMSE':<10} {'Corr':<10} {'Bias':<10} {'Status'}")
print("-"*50)

for idx, name in enumerate(param_names):
    true_vals = theta_true[:, idx]
    est_vals = theta_hat[:, idx]
    rmse = np.sqrt(np.mean((true_vals - est_vals)**2))
    corr = np.corrcoef(true_vals, est_vals)[0, 1]
    bias = np.mean(est_vals - true_vals)
    status = 'PASS' if (rmse < 0.3 and corr > 0.7) else 'CHECK'
    print(f"{name:<12} {rmse:<10.4f} {corr:<10.4f} {bias:<10.4f} {status}")

print("="*60)

## Section 5: Visualize Recovery

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (name, ax) in enumerate(zip(param_names, axes.flat)):
    true_vals = theta_true[:, idx]
    est_vals = theta_hat[:, idx]
    corr = np.corrcoef(true_vals, est_vals)[0, 1]
    rmse = np.sqrt(np.mean((true_vals - est_vals)**2))
    
    ax.scatter(true_vals, est_vals, alpha=0.15, s=5, c='steelblue')
    
    # 45-degree line
    lims = [min(true_vals.min(), est_vals.min()), max(true_vals.max(), est_vals.max())]
    ax.plot(lims, lims, 'k--', lw=2, label='Perfect recovery')
    
    ax.set_xlabel(f'True ${name.replace("_", "_{{")}}}$')
    ax.set_ylabel(f'Estimated $\\hat{{{name.replace("_", "_{{")}}}}}$')
    ax.set_title(f'${name.replace("_", "_{{")}}}$: Corr={corr:.3f}, RMSE={rmse:.3f}')
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle('Multinomial Logit: Parameter Recovery ($\\hat{\\theta}$ vs $\\theta^*$)', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()